# Notebook 02_03 — Feature Selection: XGB_Importance

Runs **XGB_Importance** across K = {4, 8, 16} × 6 classifiers × 5 seeds.
Uses the pre-computed ranking from `02_00_fs_rankings.ipynb` (`models/fs_rankings.pkl`).

**Results saved incrementally to** `results/fs_xgb_importance_raw.csv` — if this notebook crashes mid-run, just re-run it: completed (K, seed, classifier) combinations are skipped automatically.

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)
Imports OK


In [2]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
y_train = d['y_train']
FEATURE_NAMES = d['feature_names']

with open(f'{MODELS_DIR}fs_rankings.pkl', 'rb') as f:
    RANKINGS = pickle.load(f)

METHOD_NAME = 'XGB_Importance'
ranked_indices = RANKINGS[METHOD_NAME]

print(f'Method: {METHOD_NAME}')
print('Top-16 features:', [FEATURE_NAMES[i] for i in ranked_indices[:16]])

Method: XGB_Importance
Top-16 features: ['wlan.fixed.reason_code', 'radiotap.channel.freq', 'tcp.time_relative', 'tcp.dstport', 'data.len', 'ip.version', 'udp.dstport', 'radiotap.mactime', 'arp.opcode', 'frame.len', 'wlan.fc.retry', 'frame.encap_type', 'tcp.srcport', 'wlan_radio.duration', 'tcp.analysis.rto_frame', 'tcp.analysis.flags']


## Transform function (slice to top-K features by this method's ranking)

In [3]:
def transform_fn(K):
    selected = ranked_indices[:K]
    return X_train[:, selected], X_val[:, selected], X_test[:, selected], K

## Run experiment grid (resumable)

In [4]:
RESULTS_CSV = f'{RESULTS_DIR}fs_xgb_importance_raw.csv'

run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

[XGB_Importance] K=4 seed=42 DT     F1=0.3660  train=0.0s  inf=0.0001ms/sample  (1/90)
[XGB_Importance] K=4 seed=42 RF     F1=0.3732  train=1.1s  inf=0.0028ms/sample  (2/90)
[XGB_Importance] K=4 seed=42 XGB    F1=0.3705  train=1.4s  inf=0.0009ms/sample  (3/90)
[XGB_Importance] K=4 seed=42 LGBM   F1=0.3696  train=0.9s  inf=0.0016ms/sample  (4/90)
[XGB_Importance] K=4 seed=42 KNN    F1=0.1651  train=0.1s  inf=0.2204ms/sample  (5/90)
[XGB_Importance] K=4 seed=42 MLP    F1=0.3165  train=31.7s  inf=0.0017ms/sample  (6/90)
[XGB_Importance] K=4 seed=52 DT     F1=0.3666  train=0.0s  inf=0.0001ms/sample  (7/90)
[XGB_Importance] K=4 seed=52 RF     F1=0.3691  train=1.1s  inf=0.0027ms/sample  (8/90)
[XGB_Importance] K=4 seed=52 XGB    F1=0.3681  train=1.3s  inf=0.0009ms/sample  (9/90)
[XGB_Importance] K=4 seed=52 LGBM   F1=0.3687  train=0.9s  inf=0.0017ms/sample  (10/90)
[XGB_Importance] K=4 seed=52 KNN    F1=0.1652  train=0.1s  inf=0.1771ms/sample  (11/90)
[XGB_Importance] K=4 seed=52 MLP    F1=0

## Quick summary

In [5]:
res_df = pd.read_csv(RESULTS_CSV)
summary = res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std'])
print(summary.to_string())

                     f1           train_time_s            inference_ms          
                   mean       std         mean        std         mean       std
K  classifier                                                                   
4  DT          0.365691  0.001821     0.036074   0.001415     0.000060  0.000001
   KNN         0.165099  0.000084     0.074981   0.000980     0.219825  0.042998
   LGBM        0.368590  0.001323     0.881531   0.007891     0.001691  0.000044
   MLP         0.316375  0.000136    28.038032   9.155061     0.001417  0.000417
   RF          0.369589  0.003214     1.095604   0.007970     0.002745  0.000023
   XGB         0.368226  0.002446     1.421123   0.097026     0.000902  0.000014
8  DT          0.640459  0.001717     0.059036   0.004016     0.000062  0.000002
   KNN         0.452787  0.001123     0.142119   0.003318     0.096545  0.012007
   LGBM        0.285499  0.329142     1.119524   0.081554     0.001705  0.000699
   MLP         0.558307  0.0